In [1]:
!pip install -q -U num2words bitsandbytes datasets transformers tensorboard matplotlib

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
grpcio-tools 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.36.2 which is incompatible.
ydata-profiling 4.18.4 requires matplotlib<=3.10,>=3.5, but you have matplotlib 3.11.2 which is incompatible.
tensorflow 2.20.0 requires tensorboard~=2.20.0, but you have tensorboard 2.21.0 which is incompatible.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.36.2 which is incompatible.
ydf

In [3]:
%%writefile /kaggle/working/train_v2.py

"""
Full fine-tune HuggingFaceTB/SmolVLM2-500M-Video-Instruct on keplerccc/Robo2VLM-1  (Kaggle 2x T4, DDP)

Splits   : the dataset ships only `train` and `test`. We keep `train` for training and cut `test`
           into two disjoint parts:  VALID = first N_VAL rows (used for model selection),
           TEST = all remaining rows (touched once, at the very end, with the best model).
Best model: highest VALID accuracy (letter exact-match). Saved to best/ whenever it improves.
Memory   : best/ and final/ are written in fp16 (~1 GB) straight from GPU tensors, in small shards;
           last/ (fp32 + optimizer, for resuming) is written every SAVE_EVERY steps.
Launch   : torchrun --standalone --nproc_per_node=2 train_v2.py
"""
import os, io, re, ast, gc, json, csv, time, shutil, random, datetime, logging, contextlib
from itertools import islice
from pathlib import Path

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("MALLOC_ARENA_MAX", "2")          # curbs glibc RSS growth in long-running workers
os.environ.setdefault("HF_HOME", "/tmp/hf")

import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader
from PIL import Image
from datasets import load_dataset, Image as HFImage
from datasets.distributed import split_dataset_by_node
from transformers import AutoProcessor, AutoModelForImageTextToText, get_cosine_schedule_with_warmup

# ----------------------------------------------------------------------------- config
MODEL_ID = "HuggingFaceTB/SmolVLM2-500M-Video-Instruct"
DATASET = "keplerccc/Robo2VLM-1"
OUT_DIR = Path("/kaggle/working/robo2vlm_smolvlm2_run")

MAX_TRAIN_SAMPLES = 100_000      # global samples to train on (full train split = 678,034)
PER_DEVICE_BS = 4
GRAD_ACCUM = 8                   # effective batch = 4 * 2 GPUs * 8 = 64
LR = 2e-5
VISION_LR_MULT = 0.5
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 50
MAX_GRAD_NORM = 1.0
FREEZE_VISION = False            # False = true full fine-tune
IMAGE_SPLITTING = False          # False: one 512x512 tile (64 visual tokens)
OPTIM = "adamw8bit"              # "adamw8bit" or "adamw"
USE_AMP = True                   # fp16 autocast + GradScaler (T4 has no bf16)
GRAD_CKPT = True

N_VAL = 500                      # rows of `test` used as validation (model selection)
N_TEST = 0                       # rows of `test` used as final test after the N_VAL ones; 0 = all remaining
EVAL_EVERY = 250                 # validate every N optimizer steps
SAVE_EVERY = 500                 # write resumable last/ every N steps
LOG_EVERY = 10
EVAL_BS = 8
NUM_WORKERS = 1                  # per GPU process; lower to 0 if host RAM still climbs
SHUFFLE_BUFFER = 64
STREAM_BATCH = 64                # parquet record-batch rows (smaller = less host RAM); ignored if unsupported
MAX_HOURS = 10.5                 # stop + save before Kaggle's 12 h limit
SEED = 42
RESUME = True

# ----------------------------------------------------------------------------- dist / logging
rank = int(os.environ.get("RANK", 0)); world = int(os.environ.get("WORLD_SIZE", 1))
local_rank = int(os.environ.get("LOCAL_RANK", 0))
torch.cuda.set_device(local_rank); dev = torch.device("cuda", local_rank)
dist.init_process_group("nccl", timeout=datetime.timedelta(minutes=60))
is_main = rank == 0

OUT_DIR.mkdir(parents=True, exist_ok=True)
logger = logging.getLogger("train"); logger.setLevel(logging.INFO if is_main else logging.WARNING)
fmt = logging.Formatter("%(asctime)s | %(message)s", "%H:%M:%S")
sh = logging.StreamHandler(); sh.setFormatter(fmt); logger.addHandler(sh)
if is_main:
    fh = logging.FileHandler(OUT_DIR / "train.log"); fh.setFormatter(fmt); logger.addHandler(fh)
log = logger.info


def all_mean(x):
    t = torch.tensor([x], device=dev, dtype=torch.float32); dist.all_reduce(t)
    return (t / world).item()


random.seed(SEED + rank); torch.manual_seed(SEED)
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login
    login(os.environ["HF_TOKEN"])

# ----------------------------------------------------------------------------- processor
def make_processor(padding_side):
    p = AutoProcessor.from_pretrained(MODEL_ID, do_image_splitting=IMAGE_SPLITTING)
    p.image_processor.do_image_splitting = IMAGE_SPLITTING
    p.tokenizer.padding_side = padding_side
    return p

processor = make_processor("right")        # training
eval_processor = make_processor("left")    # generation
tok = processor.tokenizer
EOU_ID = tok.convert_tokens_to_ids("<end_of_utterance>")
IMG_ID = tok.convert_tokens_to_ids("<image>")

# ----------------------------------------------------------------------------- data
def parse_choices(c):
    if isinstance(c, (list, tuple)):
        out = [str(x) for x in c]
    else:
        s = str(c).strip(); out = None
        for fn in (json.loads, ast.literal_eval):
            try:
                v = fn(s)
                if isinstance(v, (list, tuple)): out = [str(x) for x in v]; break
                if isinstance(v, dict): out = [str(v[k]) for k in sorted(v)]; break
            except Exception:
                pass
        if out is None:
            out = [p.strip() for p in re.split(r"\n|;;", s) if p.strip()]
    return [re.sub(r"^\(?[A-Za-z][\.\):]\s+", "", x).strip() for x in out]


def build_prompt(question, choices):
    opts = "\n".join(f"{chr(65 + i)}. {c}" for i, c in enumerate(choices))
    return f"{str(question).strip()}\n\nOptions:\n{opts}\n\nAnswer with the letter of the correct option."


def load_img(v):
    if isinstance(v, Image.Image): img = v
    elif isinstance(v, dict) and v.get("bytes") is not None: img = Image.open(io.BytesIO(v["bytes"]))
    else: img = Image.open(v["path"])
    return img.convert("RGB")


_stream_bs_logged = False
def _load(split):
    global _stream_bs_logged
    try:
        ds = load_dataset(DATASET, split=split, streaming=True, batch_size=STREAM_BATCH)
        if not _stream_bs_logged: log(f"streaming with batch_size={STREAM_BATCH}")
    except Exception as e:
        if not _stream_bs_logged: log(f"batch_size not supported ({type(e).__name__}); using default")
        ds = load_dataset(DATASET, split=split, streaming=True)
    _stream_bs_logged = True
    return ds


def make_stream(split, seed, shuffle=True, by_node=True):
    ds = _load(split).cast_column("image", HFImage(decode=False))   # raw bytes; decode in collate
    if shuffle: ds = ds.shuffle(seed=seed, buffer_size=SHUFFLE_BUFFER)
    if by_node and world > 1: ds = split_dataset_by_node(ds, rank=rank, world_size=world)
    return ds


def detect_answer_base():
    ds = _load("test").remove_columns(["image"])
    has0 = hit_len = False
    for ex in islice(ds, 400):
        n = len(parse_choices(ex["choices"])); a = int(ex["correct_answer"])
        has0 |= a == 0; hit_len |= a == n
    return 1 if (hit_len and not has0) else 0

ANS_BASE = detect_answer_base()


def split_rows(start, stop):
    """Deterministic, identical on every rank/run: rows [start, stop) of the fixed-seed shuffled test split."""
    for i, ex in enumerate(make_stream("test", 0, shuffle=True, by_node=False)):
        if i < start: continue
        if stop and i >= stop: break
        yield ex


def compress_row(ex):
    buf = io.BytesIO(); load_img(ex["image"]).save(buf, "JPEG", quality=92)
    return {"question": ex["question"], "choices": ex["choices"], "correct_answer": ex["correct_answer"],
            "image": {"bytes": buf.getvalue()}}


def collate_train(examples):
    texts, images = [], []
    for ex in examples:
        ch = parse_choices(ex["choices"]); idx = int(ex["correct_answer"]) - ANS_BASE
        if not (0 <= idx < len(ch) <= 26): continue
        msgs = [
            {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": build_prompt(ex["question"], ch)}]},
            {"role": "assistant", "content": [{"type": "text", "text": chr(65 + idx)}]},
        ]
        texts.append(processor.apply_chat_template(msgs, add_generation_prompt=False).strip())
        images.append([load_img(ex["image"])])
    if not texts: return None
    b = processor(text=texts, images=images, return_tensors="pt", padding=True)
    ids = b["input_ids"]; labels = torch.full_like(ids, -100)
    for i in range(ids.size(0)):
        pos = (ids[i] == EOU_ID).nonzero().flatten()
        if len(pos) >= 2:   # supervise only the assistant turn
            s, e = pos[-2].item() + 1, pos[-1].item() + 1
            labels[i, s:e] = ids[i, s:e]
        else:
            labels[i] = ids[i]
    labels[labels == IMG_ID] = -100
    labels[b["attention_mask"] == 0] = -100
    b["labels"] = labels
    return b


def collate_eval(samples):
    texts, images, gold = [], [], []
    for ex in samples:
        ch = parse_choices(ex["choices"]); idx = int(ex["correct_answer"]) - ANS_BASE
        if not (0 <= idx < len(ch) <= 26): continue
        msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": build_prompt(ex["question"], ch)}]}]
        texts.append(eval_processor.apply_chat_template(msgs, add_generation_prompt=True))
        images.append([load_img(ex["image"])]); gold.append(chr(65 + idx))
    if not texts: return None, []
    return eval_processor(text=texts, images=images, return_tensors="pt", padding=True), gold


def batch_gen(start_seed):
    epoch = 0
    while True:
        dl = DataLoader(make_stream("train", start_seed + epoch), batch_size=PER_DEVICE_BS, collate_fn=collate_train,
                        num_workers=NUM_WORKERS, prefetch_factor=1 if NUM_WORKERS else None, pin_memory=False)
        for b in dl:
            if b is not None: yield b
        epoch += 1

# ----------------------------------------------------------------------------- model
last_dir = OUT_DIR / "last"
resume_state = None
if RESUME and (last_dir / "trainer_state.pt").exists():
    resume_state = torch.load(last_dir / "trainer_state.pt", map_location="cpu", weights_only=False)
    log(f"Resuming from {last_dir} at step {resume_state['step']}")
src = str(last_dir) if resume_state else MODEL_ID

model = AutoModelForImageTextToText.from_pretrained(src, attn_implementation="sdpa")
model.float(); model.config.use_cache = False
if GRAD_CKPT: model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
if FREEZE_VISION:
    for n, p in model.named_parameters():
        if "vision_model" in n: p.requires_grad = False
model.to(dev)
n_total = sum(p.numel() for p in model.parameters()); n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
log(f"params: total {n_total/1e6:.1f}M | trainable {n_train/1e6:.1f}M | GPU mem after load {torch.cuda.memory_allocated(dev)/2**30:.2f} GB")
model = DDP(model, device_ids=[local_rank], find_unused_parameters=False, broadcast_buffers=False)

# ----------------------------------------------------------------------------- optimizer / schedule
groups = {}
for n, p in model.named_parameters():
    if not p.requires_grad: continue
    vis = "vision_model" in n; nodecay = p.ndim < 2 or n.endswith(".bias")
    groups.setdefault((vis, nodecay), []).append(p)
param_groups = [{"params": ps, "lr": LR * (VISION_LR_MULT if vis else 1.0), "weight_decay": 0.0 if nd else WEIGHT_DECAY}
                for (vis, nd), ps in groups.items()]
if OPTIM == "adamw8bit":
    import bitsandbytes as bnb
    opt = bnb.optim.AdamW8bit(param_groups, betas=(0.9, 0.999))
else:
    opt = torch.optim.AdamW(param_groups, betas=(0.9, 0.999), fused=True)

STEP_SAMPLES = PER_DEVICE_BS * world * GRAD_ACCUM
MAX_STEPS = MAX_TRAIN_SAMPLES // STEP_SAMPLES
sched = get_cosine_schedule_with_warmup(opt, WARMUP_STEPS, MAX_STEPS)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
step, best_acc = 0, -1.0
if resume_state:
    opt.load_state_dict(resume_state["opt"]); sched.load_state_dict(resume_state["sched"])
    scaler.load_state_dict(resume_state["scaler"]); step = resume_state["step"]
    best_acc = resume_state.get("best_acc", -1.0)
    del resume_state
params = [p for p in model.parameters() if p.requires_grad]

# ----------------------------------------------------------------------------- bookkeeping
if is_main:
    cfg = {k: (str(v) if isinstance(v, Path) else v) for k, v in globals().items()
           if k.isupper() and isinstance(v, (int, float, str, bool, Path))}
    cfg.update(world_size=world, answer_index_base=ANS_BASE, max_steps=MAX_STEPS, effective_batch=STEP_SAMPLES,
               torch=torch.__version__, gpus=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
    (OUT_DIR / "config.json").write_text(json.dumps(cfg, indent=2))
    try:
        from torch.utils.tensorboard import SummaryWriter
        tb = SummaryWriter(str(OUT_DIR / "tb"))
    except Exception:
        tb = None
    jsonl = open(OUT_DIR / "log_history.jsonl", "a"); csv_path = OUT_DIR / "log_history.csv"
log(f"answer base={ANS_BASE} | max_steps={MAX_STEPS} | effective batch={STEP_SAMPLES} | samples={MAX_STEPS*STEP_SAMPLES}")


def record(d):
    if not is_main: return
    jsonl.write(json.dumps(d) + "\n"); jsonl.flush()
    new = not csv_path.exists()
    with open(csv_path, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(d.keys()), extrasaction="ignore")
        if new: w.writeheader()
        w.writerow(d)
    if tb:
        for k, v in d.items():
            if k != "step" and isinstance(v, (int, float)): tb.add_scalar(k, v, d["step"])


def save_weights(tag, half):
    """half=True: fp16 copy built tensor-by-tensor on GPU (no big host copy), ~1 GB on disk."""
    if is_main:
        tmp, final = OUT_DIR / f"{tag}_tmp", OUT_DIR / tag
        shutil.rmtree(tmp, ignore_errors=True)
        net = model.module
        sd = ({k: (v.detach().to(torch.float16) if v.is_floating_point() else v.detach())
               for k, v in net.state_dict().items()} if half else None)
        net.save_pretrained(tmp, state_dict=sd, safe_serialization=True, max_shard_size="2GB" if half else "500MB")
        processor.save_pretrained(tmp)
        del sd
        shutil.rmtree(final, ignore_errors=True); tmp.rename(final)
        log(f"saved {final} ({'fp16' if half else 'fp32'})")
    dist.barrier(); gc.collect(); torch.cuda.empty_cache()


def save_resume():
    if is_main:
        tmp, final = OUT_DIR / "last_tmp", OUT_DIR / "last"
        shutil.rmtree(tmp, ignore_errors=True)
        model.module.save_pretrained(tmp, safe_serialization=True, max_shard_size="500MB"); processor.save_pretrained(tmp)
        torch.save({"opt": opt.state_dict(), "sched": sched.state_dict(), "scaler": scaler.state_dict(),
                    "step": step, "best_acc": best_acc}, tmp / "trainer_state.pt")
        shutil.rmtree(final, ignore_errors=True); tmp.rename(final)
        log(f"saved {final} (resumable)")
    dist.barrier(); gc.collect(); torch.cuda.empty_cache()

# ----------------------------------------------------------------------------- eval
@torch.no_grad()
def run_eval(net, rows):
    """Greedy-decode the answer letter. Rows are split across ranks by index; results are all-reduced."""
    net.eval()
    tot, cor, invalid = [0] * 26, [0] * 26, 0
    def flush(buf):
        nonlocal invalid
        inputs, gold = collate_eval(buf)
        if not gold: return
        inputs = {k: v.to(dev) for k, v in inputs.items()}
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
            out = net.generate(**inputs, max_new_tokens=3, do_sample=False, use_cache=True)
        txt = eval_processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        for t, g in zip(txt, gold):
            m = re.search(r"[A-Z]", t.upper()); gi = ord(g) - 65
            tot[gi] += 1
            if m is None: invalid += 1
            elif m.group(0) == g: cor[gi] += 1
    buf = []
    for i, ex in enumerate(rows):
        if i % world != rank: continue
        buf.append(ex)
        if len(buf) == EVAL_BS: flush(buf); buf = []
    if buf: flush(buf)
    s = torch.tensor([invalid] + tot + cor, device=dev, dtype=torch.float64); dist.all_reduce(s)
    s = s.tolist(); inv, tot, cor = s[0], s[1:27], s[27:53]
    n = sum(tot)
    return {"acc": sum(cor) / max(n, 1), "invalid_rate": inv / max(n, 1), "n": int(n),
            "majority_baseline": max(tot) / max(n, 1),
            "acc_by_gold_letter": {chr(65 + i): round(cor[i] / tot[i], 4) for i in range(26) if tot[i]},
            "n_by_gold_letter": {chr(65 + i): int(tot[i]) for i in range(26) if tot[i]}}


log(f"building validation set ({N_VAL} rows from test split)")
val_rows = [compress_row(ex) for ex in split_rows(0, N_VAL)]

# ----------------------------------------------------------------------------- train
t0 = time.time(); t_win = time.time()
win_loss = win_n = 0.0; win_samples = 0
batches = batch_gen(SEED + step)
model.train(); opt.zero_grad(set_to_none=True)
log("starting training")

while step < MAX_STEPS:
    step_loss = 0.0
    for m in range(GRAD_ACCUM):
        batch = next(batches)
        if step == 0 and m == 0 and is_main:
            s = re.sub(r"(<image>)+", "<image...>", tok.decode(batch["input_ids"][0]))
            sup = tok.decode(batch["input_ids"][0][batch["labels"][0] != -100])
            log(f"SANITY first sample:\n{s}\n--- supervised tokens -> {sup!r}")
        batch = {k: v.to(dev, non_blocking=True) for k, v in batch.items()}
        ctx = model.no_sync() if m < GRAD_ACCUM - 1 else contextlib.nullcontext()
        with ctx:
            with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
                loss = model(**batch).loss / GRAD_ACCUM
            scaler.scale(loss).backward()
        step_loss += loss.detach().float().item()
        del batch, loss
    scaler.unscale_(opt)
    gnorm = torch.nn.utils.clip_grad_norm_(params, MAX_GRAD_NORM).item()
    scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True)
    step += 1
    win_loss += step_loss; win_n += 1; win_samples += STEP_SAMPLES

    if step % LOG_EVERY == 0 or step == MAX_STEPS:
        avg = all_mean(win_loss / win_n); now = time.time()
        rec = {"step": step, "samples_seen": step * STEP_SAMPLES, "epoch": round(step * STEP_SAMPLES / 678034, 4),
               "loss": round(avg, 5), "lr": sched.get_last_lr()[0], "grad_norm": round(gnorm, 4),
               "loss_scale": scaler.get_scale() if USE_AMP else 1.0,
               "samples_per_sec": round(win_samples / (now - t_win), 2),
               "gpu_peak_gb": round(torch.cuda.max_memory_allocated(dev) / 2**30, 2),
               "elapsed_min": round((now - t0) / 60, 1)}
        log(" | ".join(f"{k}={v}" for k, v in rec.items()))
        record(rec); win_loss = win_n = 0.0; win_samples = 0; t_win = now

    if step % EVAL_EVERY == 0 or step == MAX_STEPS:
        r = run_eval(model.module, val_rows); model.train()
        improved = r["acc"] > best_acc
        log(f"VALID step {step}: acc={r['acc']:.4f} (majority baseline {r['majority_baseline']:.3f}) "
            f"invalid={r['invalid_rate']:.3f} n={r['n']} {'<- new best' if improved else ''}")
        record({"step": step, "val_acc": round(r["acc"], 5), "val_invalid_rate": round(r["invalid_rate"], 5), "val_n": r["n"]})
        if improved:
            best_acc = r["acc"]; save_weights("best", half=True)

    stop = torch.tensor([float(is_main and (time.time() - t0) > MAX_HOURS * 3600)], device=dev)
    dist.all_reduce(stop, op=dist.ReduceOp.MAX)
    if step % SAVE_EVERY == 0 or stop.item() > 0:
        save_resume()
    if stop.item() > 0:
        log("time budget reached -> stopping (re-run the same cell to resume)"); break

# ----------------------------------------------------------------------------- finish: final test with BEST model
save_weights("final", half=True)
if is_main:
    jsonl.flush()
del opt, scaler, sched, params, groups, param_groups, model
gc.collect(); torch.cuda.empty_cache()

best_dir = OUT_DIR / ("best" if (OUT_DIR / "best").exists() else "final")
log(f"final TEST with weights from {best_dir.name}/")
test_net = AutoModelForImageTextToText.from_pretrained(str(best_dir), attn_implementation="sdpa").float().to(dev).eval()
test_res = run_eval(test_net, split_rows(N_VAL, (N_VAL + N_TEST) if N_TEST else 0))
test_res.update(weights=best_dir.name, best_val_acc=best_acc)
log(f"TEST: acc={test_res['acc']:.4f} (majority baseline {test_res['majority_baseline']:.3f}) "
    f"invalid={test_res['invalid_rate']:.3f} n={test_res['n']}")
if is_main:
    (OUT_DIR / "test_results.json").write_text(json.dumps(test_res, indent=2))
    record({"step": step, "test_acc": round(test_res["acc"], 5), "test_n": test_res["n"]})
    hist = [json.loads(l) for l in open(OUT_DIR / "log_history.jsonl")]
    (OUT_DIR / "history.json").write_text(json.dumps({"config": cfg, "log_history": hist, "test": test_res}, indent=2))
    try:
        import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
        tr = [h for h in hist if "loss" in h]; ev = [h for h in hist if "val_acc" in h]
        fig, ax = plt.subplots(1, 3, figsize=(16, 4))
        ax[0].plot([h["step"] for h in tr], [h["loss"] for h in tr]); ax[0].set_title("train loss")
        ax[1].plot([h["step"] for h in ev], [h["val_acc"] for h in ev], marker="o"); ax[1].set_title("validation accuracy")
        ax[2].plot([h["step"] for h in tr], [h["lr"] for h in tr]); ax[2].set_title("lr")
        for a in ax: a.set_xlabel("step"); a.grid(alpha=.3)
        plt.tight_layout(); plt.savefig(OUT_DIR / "curves.png", dpi=120)
    except Exception as e:
        log(f"plot failed: {e}")
    if tb: tb.close()
    jsonl.close()
    log(f"DONE. best val acc={best_acc:.4f} | test acc={test_res['acc']:.4f} | outputs in {OUT_DIR}")
dist.barrier(); dist.destroy_process_group()

Writing /kaggle/working/train_v2.py


In [ ]:
!rm -rf /kaggle/working/robo2vlm_smolvlm2_run   # remove this line if you want to RESUME
!mkdir -p /kaggle/working/robo2vlm_smolvlm2_run
!cd /kaggle/working && torchrun --standalone --nproc_per_node=2 train_v2.py 2>&1 | tee -a /kaggle/working/robo2vlm_smolvlm2_run/stdout.log

W1031 03:25:28.062000 172 torch/distributed/run.py:851] 
W1031 03:25:28.062000 172 torch/distributed/run.py:851] *****************************************
W1031 03:25:28.062000 172 torch/distributed/run.py:851] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1031 03:25:28.062000 172 torch/distributed/run.py:851] *****************************************
[transformers] Image processor SmolVLMVideoProcessor: kwargs ['do_image_splitting'] were applied for backward compatibility. To avoid this warning, add them to valid_kwargs: create a custom TypedDict extending ImagesKwargs with these keys and set it as the `valid_kwargs` class attribute.
[transformers] Image processor SmolVLMVideoProcessor: kwargs ['do_image_splitting'] were applied for backward compatibility. To avoid this warning, add them to valid_kwargs: create a custom Ty

In [ ]:
!ls -lh /kaggle/working/robo2vlm_smolvlm2_run /kaggle/working/robo2vlm_smolvlm2_run/best
!cat /kaggle/working/robo2vlm_smolvlm2_run/test_results.json
from IPython.display import Image, display
import os
p='/kaggle/working/robo2vlm_smolvlm2_run/curves.png'
if os.path.exists(p): display(Image(p))